# Inspect Processed ABIDE SINDy Dataset

This notebook loads the processed export created by `Data_processing.py` and lets you inspect each sample and its window-level simplicial information.

In [4]:
from pathlib import Path
from pprint import pprint
import numpy as np
import pandas as pd

from Data_processing import load_processed_dataset

In [9]:
import importlib
from nilearn import datasets
import Data_processing as dp

# Ensure the notebook uses the latest Data_processing.py edits.
importlib.reload(dp)

# Process exactly one raw ABIDE subject and inspect the resulting simplicial windows.
raw_abide = datasets.fetch_abide_pcp(
    data_dir='./ABIDE_pcp',
    derivatives='rois_cc200',
    n_subjects=1,
    verbose=1,
)

processor = dp.ABIDESimplicialProcessor(
    data_dir='./ABIDE_pcp',
    save_intermediates=False,
    upsample_factor=100,
)

single_subject_ts = raw_abide.rois_cc200[0]
single_subject_id = processor._get_subject_id(raw_abide, 0)
processed_one = processor.process_subject(
    subject_ts=single_subject_ts,
    window_len=1024,
    window_overlap=0.5,
    subject_id=single_subject_id,
)

print(f"subject_id: {processed_one.get('subject_id', 'unknown')}")
print(f"n_windows: {processed_one.get('n_windows', 0)}")
print(f"error: {processed_one.get('error', 'none')}")

windows = processed_one.get('windows', [])
if windows:
    first_window = windows[0]
    print('\n=== First Window Shapes ===')
    print(f"node_features: {np.asarray(first_window.node_features).shape}")
    print(f"edge_features: {np.asarray(first_window.edge_features).shape}")
    print(f"triangle_features: {np.asarray(first_window.triangle_features).shape}")
    print(f"edges: {np.asarray(first_window.incidence['edges']).shape}")
    print(f"triangles: {np.asarray(first_window.incidence['triangles']).shape}")
    print(f"B1: {np.asarray(first_window.incidence['B1']).shape}")
    print(f"B2: {np.asarray(first_window.incidence['B2']).shape}")

    preview_rows = []
    for idx, window in enumerate(windows[:10]):
        preview_rows.append({
            'window_idx': idx,
            'node_shape': tuple(np.asarray(window.node_features).shape),
            'edge_shape': tuple(np.asarray(window.edge_features).shape),
            'triangle_shape': tuple(np.asarray(window.triangle_features).shape),
            'n_edges': int(window.incidence.get('n_edges', 0)),
            'n_triangles': int(window.incidence.get('n_triangles', 0)),
        })

    print('\n=== Window Preview ===')
    display(pd.DataFrame(preview_rows))
else:
    print('No windows were created for this subject.')

[fetch_abide_pcp] Dataset found in ABIDE_pcp/ABIDE_pcp
subject_id: subject_0000
n_windows: 37
error: none

=== First Window Shapes ===
node_features: (200, 1024)
edge_features: (141, 1024)
triangle_features: (10, 1024)
edges: (141, 2)
triangles: (10, 3)
B1: (200, 141)
B2: (141, 10)

=== Window Preview ===


,window_idx,node_shape,edge_shape,triangle_shape,n_edges,n_triangles
0,0,"(200, 1024)","(141, 1024)","(10, 1024)",141,10
1,1,"(200, 1024)","(40, 1024)","(0, 1024)",40,0
2,2,"(200, 1024)","(166, 1024)","(15, 1024)",166,15
3,3,"(200, 1024)","(91, 1024)","(1, 1024)",91,1
4,4,"(200, 1024)","(30, 1024)","(0, 1024)",30,0
5,5,"(200, 1024)","(48, 1024)","(0, 1024)",48,0
6,6,"(200, 1024)","(108, 1024)","(14, 1024)",108,14
7,7,"(200, 1024)","(86, 1024)","(3, 1024)",86,3
8,8,"(200, 1024)","(94, 1024)","(21, 1024)",94,21
9,9,"(200, 1024)","(11, 1024)","(0, 1024)",11,0


In [10]:
# Validate required rank-based feature/incidence/adjacency payloads.
def validate_window_matrices(window):
    required = {
        'features': ['rank_0', 'rank_1', 'rank_2'],
        'incidences': ['rank_1', 'rank_2'],
        'adjacencies': ['rank_0', 'rank_1', 'rank_2'],
    }

    report = {
        'missing': {'features': [], 'incidences': [], 'adjacencies': []},
        'shapes': {}
    }

    # 1) Validate rank-based dictionaries introduced by preprocessing update.
    features = getattr(window, 'features', None)
    if isinstance(features, dict):
        for key in required['features']:
            if key in features:
                report['shapes'][f'features.{key}'] = tuple(np.asarray(features[key]).shape)
            else:
                report['missing']['features'].append(key)
    else:
        report['missing']['features'].extend(required['features'])

    incidences = getattr(window, 'incidences', None)
    if isinstance(incidences, dict):
        for key in required['incidences']:
            if key in incidences:
                report['shapes'][f'incidences.{key}'] = tuple(np.asarray(incidences[key]).shape)
            else:
                report['missing']['incidences'].append(key)
    else:
        report['missing']['incidences'].extend(required['incidences'])

    adjacencies = getattr(window, 'adjacencies', None)
    if isinstance(adjacencies, dict):
        for key in required['adjacencies']:
            if key in adjacencies:
                report['shapes'][f'adjacencies.{key}'] = tuple(np.asarray(adjacencies[key]).shape)
            else:
                report['missing']['adjacencies'].append(key)
    else:
        report['missing']['adjacencies'].extend(required['adjacencies'])

    # 2) Also surface legacy aliases for compatibility inspection.
    legacy_incidence = getattr(window, 'incidence', None)
    if isinstance(legacy_incidence, dict):
        for key in ['B1', 'B2', 'edges', 'triangles', 'n_edges', 'n_triangles']:
            if key in legacy_incidence:
                value = legacy_incidence[key]
                if key in {'n_edges', 'n_triangles'}:
                    report['shapes'][f'legacy.{key}'] = int(value)
                else:
                    report['shapes'][f'legacy.{key}'] = tuple(np.asarray(value).shape)

    legacy_adjacency = getattr(window, 'adjacency', None)
    if isinstance(legacy_adjacency, dict):
        for key in ['H0_up', 'H1_down', 'H1_up', 'H2_down']:
            if key in legacy_adjacency:
                report['shapes'][f'legacy.{key}'] = tuple(np.asarray(legacy_adjacency[key]).shape)

    # Keep direct feature arrays visible too.
    for key in ['node_features', 'edge_features', 'triangle_features']:
        if hasattr(window, key):
            report['shapes'][f'legacy.{key}'] = tuple(np.asarray(getattr(window, key)).shape)

    return report


def validate_windows_collection(windows_to_check, max_windows=5):
    if not windows_to_check:
        print('No windows provided for validation.')
        return pd.DataFrame([])

    rows = []
    for w_idx, window in enumerate(windows_to_check[:max_windows]):
        report = validate_window_matrices(window)
        rows.append({
            'window_idx': w_idx,
            'missing_features': len(report['missing']['features']),
            'missing_incidences': len(report['missing']['incidences']),
            'missing_adjacencies': len(report['missing']['adjacencies']),
            'features.rank_0': report['shapes'].get('features.rank_0', None),
            'features.rank_1': report['shapes'].get('features.rank_1', None),
            'features.rank_2': report['shapes'].get('features.rank_2', None),
            'incidences.rank_1': report['shapes'].get('incidences.rank_1', None),
            'incidences.rank_2': report['shapes'].get('incidences.rank_2', None),
            'adjacencies.rank_0': report['shapes'].get('adjacencies.rank_0', None),
            'adjacencies.rank_1': report['shapes'].get('adjacencies.rank_1', None),
            'adjacencies.rank_2': report['shapes'].get('adjacencies.rank_2', None),
            'legacy.B1': report['shapes'].get('legacy.B1', None),
            'legacy.B2': report['shapes'].get('legacy.B2', None),
            'legacy.H0_up': report['shapes'].get('legacy.H0_up', None),
            'legacy.H1_down': report['shapes'].get('legacy.H1_down', None),
            'legacy.H1_up': report['shapes'].get('legacy.H1_up', None),
            'legacy.H2_down': report['shapes'].get('legacy.H2_down', None),
            'legacy.n_edges': report['shapes'].get('legacy.n_edges', None),
            'legacy.n_triangles': report['shapes'].get('legacy.n_triangles', None),
            'missing_detail': report['missing'],
        })

    out = pd.DataFrame(rows)
    all_ok = (
        (out['missing_features'] == 0)
        & (out['missing_incidences'] == 0)
        & (out['missing_adjacencies'] == 0)
    ).all()

    print(f'Checked {len(out)} window(s). All required rank payloads present: {bool(all_ok)}')
    return out

# Validate the one-subject run from the earlier cell.
validation_df = validate_windows_collection(windows, max_windows=10)
validation_df

Checked 10 window(s). All required rank payloads present: True


,window_idx,missing_features,missing_incidences,missing_adjacencies,features.rank_0,features.rank_1,features.rank_2,incidences.rank_1,incidences.rank_2,adjacencies.rank_0,...,adjacencies.rank_2,legacy.B1,legacy.B2,legacy.H0_up,legacy.H1_down,legacy.H1_up,legacy.H2_down,legacy.n_edges,legacy.n_triangles,missing_detail
0,0,0,0,0,"(200, 1024)","(141, 1024)","(10, 1024)","(200, 141)","(141, 10)","(200, 200)",...,"(10, 10)","(200, 141)","(141, 10)","(200, 200)","(141, 141)","(141, 141)","(10, 10)",141,10,"{'features': [], 'incidences': [], 'adjacencie..."
1,1,0,0,0,"(200, 1024)","(40, 1024)","(0, 1024)","(200, 40)","(40, 0)","(200, 200)",...,"(0, 0)","(200, 40)","(40, 0)","(200, 200)","(40, 40)","(40, 40)","(0, 0)",40,0,"{'features': [], 'incidences': [], 'adjacencie..."
2,2,0,0,0,"(200, 1024)","(166, 1024)","(15, 1024)","(200, 166)","(166, 15)","(200, 200)",...,"(15, 15)","(200, 166)","(166, 15)","(200, 200)","(166, 166)","(166, 166)","(15, 15)",166,15,"{'features': [], 'incidences': [], 'adjacencie..."
3,3,0,0,0,"(200, 1024)","(91, 1024)","(1, 1024)","(200, 91)","(91, 1)","(200, 200)",...,"(1, 1)","(200, 91)","(91, 1)","(200, 200)","(91, 91)","(91, 91)","(1, 1)",91,1,"{'features': [], 'incidences': [], 'adjacencie..."
4,4,0,0,0,"(200, 1024)","(30, 1024)","(0, 1024)","(200, 30)","(30, 0)","(200, 200)",...,"(0, 0)","(200, 30)","(30, 0)","(200, 200)","(30, 30)","(30, 30)","(0, 0)",30,0,"{'features': [], 'incidences': [], 'adjacencie..."
5,5,0,0,0,"(200, 1024)","(48, 1024)","(0, 1024)","(200, 48)","(48, 0)","(200, 200)",...,"(0, 0)","(200, 48)","(48, 0)","(200, 200)","(48, 48)","(48, 48)","(0, 0)",48,0,"{'features': [], 'incidences': [], 'adjacencie..."
6,6,0,0,0,"(200, 1024)","(108, 1024)","(14, 1024)","(200, 108)","(108, 14)","(200, 200)",...,"(14, 14)","(200, 108)","(108, 14)","(200, 200)","(108, 108)","(108, 108)","(14, 14)",108,14,"{'features': [], 'incidences': [], 'adjacencie..."
7,7,0,0,0,"(200, 1024)","(86, 1024)","(3, 1024)","(200, 86)","(86, 3)","(200, 200)",...,"(3, 3)","(200, 86)","(86, 3)","(200, 200)","(86, 86)","(86, 86)","(3, 3)",86,3,"{'features': [], 'incidences': [], 'adjacencie..."
8,8,0,0,0,"(200, 1024)","(94, 1024)","(21, 1024)","(200, 94)","(94, 21)","(200, 200)",...,"(21, 21)","(200, 94)","(94, 21)","(200, 200)","(94, 94)","(94, 94)","(21, 21)",94,21,"{'features': [], 'incidences': [], 'adjacencie..."
9,9,0,0,0,"(200, 1024)","(11, 1024)","(0, 1024)","(200, 11)","(11, 0)","(200, 200)",...,"(0, 0)","(200, 11)","(11, 0)","(200, 200)","(11, 11)","(11, 11)","(0, 0)",11,0,"{'features': [], 'incidences': [], 'adjacencie..."


In [2]:
# Change this if your processed dataset lives elsewhere.
processed_dir = Path('./abide_simplicial_dataset')

samples, labels, splits, metadata = load_processed_dataset(
    output_dir=str(processed_dir),
    allow_npz_fallback=True,
)

print(f'Loaded {len(samples)} samples from: {processed_dir.resolve()}')

Loaded 871 samples from: /hdfs1/Data/Shubhajit/Project/New_TETON/abide_simplicial_dataset


In [3]:
print('=== Split Sizes ===')
for k, v in splits.items():
    print(f'{k}: {len(v)}')

print('\n=== Label Distribution ===')
unique, counts = np.unique(np.asarray(labels), return_counts=True)
for u, c in zip(unique, counts):
    print(f'label {int(u)}: {int(c)}')

print('\n=== Metadata ===')
pprint(metadata)

=== Split Sizes ===
train: 609
val: 130
test: 132

=== Label Distribution ===
label 0: 468
label 1: 403

=== Metadata ===
{'_processed_cache_source': 'fast_pickle',
 'label_distribution': [468, 403],
 'labels': [1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            1,
            0,
            1,
            1,
            0,
            0,
            0,
            0,
            0,
     

In [4]:
def _window_stats(window):
    node_shape = tuple(np.asarray(getattr(window, 'node_features', np.array([]))).shape)
    edge_shape = tuple(np.asarray(getattr(window, 'edge_features', np.array([]))).shape)
    tri_shape = tuple(np.asarray(getattr(window, 'triangle_features', np.array([]))).shape)

    incidence = getattr(window, 'incidence', {}) if hasattr(window, 'incidence') else {}
    n_edges = incidence.get('n_edges', edge_shape[0] if len(edge_shape) > 0 else 0)
    n_triangles = incidence.get('n_triangles', tri_shape[0] if len(tri_shape) > 0 else 0)

    return {
        'node_shape': node_shape,
        'edge_shape': edge_shape,
        'triangle_shape': tri_shape,
        'n_edges': int(n_edges),
        'n_triangles': int(n_triangles),
    }


def summarize_sample(sample, label, sample_index=0, max_windows_to_show=5):
    windows = sample.get('windows', [])
    subject_id = sample.get('subject_id', 'unknown')

    print(f'=== Sample {sample_index} ===')
    print(f'subject_id: {subject_id}')
    print(f'label: {int(label)}')
    print(f'n_windows: {len(windows)}')

    if len(windows) == 0:
        print('No windows available for this sample.')
        return

    show_n = min(max_windows_to_show, len(windows))
    print(f'\nShowing first {show_n} window(s):')
    for w_idx in range(show_n):
        stats = _window_stats(windows[w_idx])
        print(f'  window {w_idx}: {stats}')

In [5]:
# Pick any sample index to inspect deeply.
sample_index = 0
summarize_sample(samples[sample_index], labels[sample_index], sample_index=sample_index, max_windows_to_show=8)

=== Sample 0 ===
subject_id: subject_0000
label: 1
n_windows: 25

Showing first 8 window(s):
  window 0: {'node_shape': (200,), 'edge_shape': (329, 1), 'triangle_shape': (163, 1), 'n_edges': 329, 'n_triangles': 163}
  window 1: {'node_shape': (200,), 'edge_shape': (26, 1), 'triangle_shape': (0, 1), 'n_edges': 26, 'n_triangles': 0}
  window 2: {'node_shape': (200,), 'edge_shape': (75, 1), 'triangle_shape': (1, 1), 'n_edges': 75, 'n_triangles': 1}
  window 3: {'node_shape': (200,), 'edge_shape': (184, 1), 'triangle_shape': (39, 1), 'n_edges': 184, 'n_triangles': 39}
  window 4: {'node_shape': (200,), 'edge_shape': (6, 1), 'triangle_shape': (0, 1), 'n_edges': 6, 'n_triangles': 0}
  window 5: {'node_shape': (200,), 'edge_shape': (61, 1), 'triangle_shape': (7, 1), 'n_edges': 61, 'n_triangles': 7}
  window 6: {'node_shape': (200,), 'edge_shape': (337, 1), 'triangle_shape': (246, 1), 'n_edges': 337, 'n_triangles': 246}
  window 7: {'node_shape': (200,), 'edge_shape': (225, 1), 'triangle_shape

In [7]:
samples[sample_index].get('windows', [])

In [8]:
# Build a quick table so you can scan all samples.
rows = []
for i, (sample, label) in enumerate(zip(samples, labels)):
    windows = sample.get('windows', [])
    subject_id = sample.get('subject_id', 'unknown')

    if windows:
        win_stats = [_window_stats(w) for w in windows]
        n_nodes_min = min(s['node_shape'][0] if len(s['node_shape']) > 0 else 0 for s in win_stats)
        n_nodes_max = max(s['node_shape'][0] if len(s['node_shape']) > 0 else 0 for s in win_stats)
        n_edges_min = min(s['n_edges'] for s in win_stats)
        n_edges_max = max(s['n_edges'] for s in win_stats)
        n_tri_min = min(s['n_triangles'] for s in win_stats)
        n_tri_max = max(s['n_triangles'] for s in win_stats)
    else:
        n_nodes_min = n_nodes_max = 0
        n_edges_min = n_edges_max = 0
        n_tri_min = n_tri_max = 0

    rows.append({
        'sample_idx': i,
        'subject_id': subject_id,
        'label': int(label),
        'n_windows': len(windows),
        'nodes_min': n_nodes_min,
        'nodes_max': n_nodes_max,
        'edges_min': n_edges_min,
        'edges_max': n_edges_max,
        'triangles_min': n_tri_min,
        'triangles_max': n_tri_max,
    })

summary_df = pd.DataFrame(rows)
summary_df.head(20)

,sample_idx,subject_id,label,n_windows,nodes_min,nodes_max,edges_min,edges_max,triangles_min,triangles_max
0,0,subject_0000,1,25,200,200,6,337,0,251
1,1,subject_0001,1,25,200,200,1,501,0,491
2,2,subject_0002,1,25,200,200,5,569,0,1011
3,3,subject_0003,1,25,200,200,2,407,0,314
4,4,subject_0004,1,25,200,200,3,499,0,406
5,5,subject_0005,1,25,200,200,5,323,0,218
6,6,subject_0006,1,25,200,200,16,449,0,361
7,7,subject_0007,1,25,200,200,7,495,0,467
8,8,subject_0008,1,25,200,200,10,690,0,2495
9,9,subject_0009,1,25,200,200,1,392,0,377


In [9]:
# Optional: find suspicious samples (e.g., zero windows or no edges).
suspicious = summary_df[(summary_df['n_windows'] == 0) | (summary_df['edges_max'] == 0)]
print(f'Suspicious samples: {len(suspicious)}')
suspicious

Suspicious samples: 0


,sample_idx,subject_id,label,n_windows,nodes_min,nodes_max,edges_min,edges_max,triangles_min,triangles_max
